Note: the actual training of the model will take 2+ hours. Be careful when hitting "run all".

In [ ]:
!wget https://zenodo.org/records/1290737/files/MTG/JAAH-v0.1.zip # ONLY run this if you did not upload the file to Colab yet. It will take 3-7 minutes to download if you run this.
!unzip JAAH-v0.1.zip
!pip install jams pretty_midi music21 librosa

--2026-04-22 07:02:38--  https://zenodo.org/records/1290737/files/MTG/JAAH-v0.1.zip
Resolving zenodo.org (zenodo.org)... 137.138.52.235, 137.138.153.219, 188.184.98.114, ...
Connecting to zenodo.org (zenodo.org)|137.138.52.235|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 316282970 (302M) [application/octet-stream]
Saving to: ‘JAAH-v0.1.zip’

JAAH-v0.1.zip       100%[===================>] 301.63M  17.5MB/s    in 18s     

2026-04-22 07:02:57 (16.4 MB/s) - ‘JAAH-v0.1.zip’ saved [316282970/316282970]

Archive:  JAAH-v0.1.zip
7686b918e544a20874bd5c6e51b9436cd96063f1
   creating: MTG-JAAH-7686b91/
 extracting: MTG-JAAH-7686b91/.gitignore  
  inflating: MTG-JAAH-7686b91/README.md  
   creating: MTG-JAAH-7686b91/annotations/
  inflating: MTG-JAAH-7686b91/annotations/airegin.json  
  inflating: MTG-JAAH-7686b91/annotations/all_alone.json  
  inflating: MTG-JAAH-7686b91/annotations/bags_groove.json  
  inflating: MTG-JAAH-7686b91/annotations/big_butter_and_eggman.js

In [ ]:
import os, json, glob
import numpy as np
import tensorflow as tf
from collections import Counter
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import Callback

In [ ]:
data_glob = "MTG-JAAH-7686b91/annotations/*.json"

jams_files = sorted(glob.glob(data_glob))
print("Total # of JSON files:", len(jams_files))
print("Example file:", jams_files[0] if jams_files else "NONE")

Total # of JSON files: 113
Example file: MTG-JAAH-7686b91/annotations/airegin.json


In [ ]:
import os, random, numpy as np, tensorflow as tf

SEED = 67
os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
# tf.config.experimental.enable_op_determinism() # only run if using T4 GPU

In [ ]:
def extract_all_chords_from_file(jams_file):
    with open(jams_file, "r") as f:
        jam = json.load(f)

    tokens = []
    for part in jam.get("parts", []):
        for chord_string in part.get("chords", []):
            bars = chord_string.split("|")
            for bar in bars:
                bar = bar.strip()
                if bar:
                    tokens.extend(bar.split())
    return tokens

all_chord_sequences = []
for jf in jams_files:
    seq = extract_all_chords_from_file(jf)
    if len(seq) > 0:
        all_chord_sequences.append(seq)

print("Songs loaded:", len(all_chord_sequences))
if all_chord_sequences:
    print("Example song length (# of chords):", len(all_chord_sequences[0]))
    print("Example chords:", all_chord_sequences[0][:20])

Songs loaded: 113
Example song length (# of chords): 282
Example chords: ['F:(b3,5,b7,11)', 'F:(b3,5,b7,11)', 'F:(b3,5,b7,11)', 'Eb:(b3,5,b7,11)', 'Eb:(b3,5,b7,11)', 'F:(b3,5,b7,11)', 'F:(b3,5,b7,11)', 'F:(b3,5,b7,11)', 'C:(b3,5,b7,11)', 'C:(b3,5,b7,11)', 'F:(b3,5,b7,11)', 'F:(b3,5,b7,11)', 'F:(b3,5,b7,11)', 'Eb:(b3,5,b7,11)', 'Eb:(b3,5,b7,11)', 'F:(b3,5,b7,11)', 'F:min7', 'F:min', 'C:(3,5,b7,#9)', 'F:min']


In [ ]:
import json
import re

def normalize_key(k):
    if k is None:
        return "UNK"
    k = str(k).strip()
    if not k:
        return "UNK"
    return k

song_keys = []  # index = song_id

for jf in jams_files:
    with open(jf, "r") as f:
        jam = json.load(f)
    key = normalize_key(jam.get("sandbox", {}).get("key"))
    song_keys.append(key)

print("Loaded keys for songs:", len(song_keys))
print("Example keys:", song_keys[:10])
print("Unique keys:", len(set(song_keys)))

Loaded keys for songs: 113
Example keys: ['Ab', 'C', 'F', 'G', 'Bb:min', "['Bb']", 'Bb', 'A', 'Bb', 'Eb']
Unique keys: 26


In [ ]:
def split_chord_label(chord):

    # Quality labels: maj/min/dom/dim/hdim/aug/sus/other/N
    # Extension labels: none/7/maj7/9/11/13/other/N

    if chord == "N":
        return "N", "N", "N"

    if ":" not in chord:
        return chord, "maj", "none"

    root, rest = chord.split(":", 1)

    # Checking quality
    if rest.startswith("maj"):
        quality = "maj"
    elif rest.startswith("min"):
        quality = "min"
    elif rest.startswith("hdim"):
        quality = "hdim"
    elif rest.startswith("dim"):
        quality = "dim"
    elif rest.startswith("aug"):
        quality = "aug"
    elif rest.startswith("sus"):
        quality = "sus"
    elif rest.startswith(("7", "9", "11", "13")):
        quality = "dom"
    elif rest.startswith("("):
        quality = "dom" if "b7" in rest else "other"
    else:
        quality = "other"

    # Checking extension
    if "maj7" in rest:
        ext = "maj7"
    elif "13" in rest:
        ext = "13"
    elif "11" in rest:
        ext = "11"
    elif "9" in rest:
        ext = "9"
    elif "7" in rest:
        ext = "7"
    elif any(x in rest for x in ["b9", "#9", "#11", "b13", "#5", "b5", "alt"]):
        ext = "other"
    else:
        ext = "none"

    return root, quality, ext

In [ ]:
import re

NOTE_TO_PC = {
    "C": 0, "B#": 0,
    "C#": 1, "Db": 1,
    "D": 2,
    "D#": 3, "Eb": 3,
    "E": 4, "Fb": 4,
    "F": 5, "E#": 5,
    "F#": 6, "Gb": 6,
    "G": 7,
    "G#": 8, "Ab": 8,
    "A": 9,
    "A#": 10, "Bb": 10,
    "B": 11, "Cb": 11,
}

PC_TO_NOTE = {
    0: "C",
    1: "C#",
    2: "D",
    3: "Eb",
    4: "E",
    5: "F",
    6: "F#",
    7: "G",
    8: "Ab",
    9: "A",
    10: "Bb",
    11: "B",
}

def key_to_pc(key_str):
    if key_str is None:
        return None
    s = str(key_str).strip()
    if not s:
        return None
    m = re.match(r"^([A-G](?:#|b)?)", s)
    if not m:
        return None
    return NOTE_TO_PC.get(m.group(1), None)

def extract_root_token(chord_str):
    s = str(chord_str).strip()
    if s == "N":
        return "N"
    m = re.match(r"^([A-G](?:#|b)?)", s)
    return m.group(1) if m else "N"

REL_N_CLASS = 12

In [ ]:
def transpose_chord(chord_str, shift):
    if chord_str == "N":
        return "N"

    root_tok = extract_root_token(chord_str)
    if root_tok == "N":
        return chord_str

    root_pc = NOTE_TO_PC.get(root_tok, None)
    if root_pc is None:
        return chord_str

    new_pc = (root_pc + shift) % 12
    new_root = PC_TO_NOTE[new_pc]

    rest = chord_str[len(root_tok):]
    return new_root + rest

def transpose_key(key_str, shift):
    pc = key_to_pc(key_str)
    if pc is None:
        return key_str

    new_pc = (pc + shift) % 12
    return PC_TO_NOTE[new_pc]

In [ ]:
def chord_to_rel_root(chord_str, key_tok):
    rt = extract_root_token(chord_str)
    if rt == "N":
        return REL_N_CLASS

    root_pc = NOTE_TO_PC.get(rt, None)
    key_pc = key_to_pc(key_tok)

    if root_pc is None or key_pc is None:
        return REL_N_CLASS

    return (root_pc - key_pc) % 12


def build_fold_data_from_lists(
    train_sequences,
    train_keys,
    val_song_ids,
    WINDOW
):

    X_train_tokens, y_train_chords, key_train_tokens = [], [], []

    for seq, key in zip(train_sequences, train_keys):
        if len(seq) <= WINDOW:
            continue
        for i in range(len(seq) - WINDOW):
            X_train_tokens.append(seq[i:i+WINDOW])
            y_train_chords.append(seq[i+WINDOW])
            key_train_tokens.append(key)

    X_val_tokens, y_val_chords, key_val_tokens = [], [], []

    for sid in val_song_ids:
        seq = all_chord_sequences[sid]
        key = song_keys[sid]

        if len(seq) <= WINDOW:
            continue
        for i in range(len(seq) - WINDOW):
            X_val_tokens.append(seq[i:i+WINDOW])
            y_val_chords.append(seq[i+WINDOW])
            key_val_tokens.append(key)

    special = ["<PAD>", "<UNK>"]
    train_flat = [c for w in X_train_tokens for c in w] + y_train_chords
    chord_vocab = special + sorted(set(train_flat))
    chord_to_idx = {c:i for i,c in enumerate(chord_vocab)}

    def encode_windows(X_tokens):
        X = np.zeros((len(X_tokens), WINDOW), dtype=np.int32)
        for i, w in enumerate(X_tokens):
            for t, c in enumerate(w):
                X[i, t] = chord_to_idx.get(c, chord_to_idx["<UNK>"])
        return X

    X_train = encode_windows(X_train_tokens)
    X_val   = encode_windows(X_val_tokens)


    key_vocab = ["<UNK>"] + sorted(set(key_train_tokens))
    key_to_idx = {k:i for i,k in enumerate(key_vocab)}

    key_train = np.array(
        [key_to_idx.get(k, 0) for k in key_train_tokens],
        dtype=np.int32
    )

    key_val = np.array(
        [key_to_idx.get(k, 0) for k in key_val_tokens],
        dtype=np.int32
    )

    y_root_train = np.array(
        [chord_to_rel_root(c, k) for c, k in zip(y_train_chords, key_train_tokens)],
        dtype=np.int32
    )

    y_root_val = np.array(
        [chord_to_rel_root(c, k) for c, k in zip(y_val_chords, key_val_tokens)],
        dtype=np.int32
    )


    yq_train, ye_train = [], []
    yq_val,   ye_val   = [], []

    for c in y_train_chords:
        _, q, e = split_chord_label(c)
        yq_train.append(q)
        ye_train.append(e)

    for c in y_val_chords:
        _, q, e = split_chord_label(c)
        yq_val.append(q)
        ye_val.append(e)

    qual_vocab = ["<UNK>"] + sorted(set(yq_train))
    ext_vocab  = ["<UNK>"] + sorted(set(ye_train))

    qual_to_idx = {q:i for i,q in enumerate(qual_vocab)}
    ext_to_idx  = {e:i for i,e in enumerate(ext_vocab)}

    y_quality_train = np.array(
        [qual_to_idx.get(q, 0) for q in yq_train],
        dtype=np.int32
    )

    y_quality_val = np.array(
        [qual_to_idx.get(q, 0) for q in yq_val],
        dtype=np.int32
    )

    y_ext_train = np.array(
        [ext_to_idx.get(e, 0) for e in ye_train],
        dtype=np.int32
    )

    y_ext_val = np.array(
        [ext_to_idx.get(e, 0) for e in ye_val],
        dtype=np.int32
    )

    meta = {
        "chord_vocab_size": len(chord_vocab),
        "key_vocab_size": len(key_vocab),
        "num_qualities": len(qual_vocab),
        "num_exts": len(ext_vocab),
        "num_roots": 13,
    }

    return (
        X_train, key_train, y_root_train, y_quality_train, y_ext_train,
        X_val, key_val, y_root_val, y_quality_val, y_ext_val,
        meta, chord_vocab, key_vocab
    )

In [ ]:
from tensorflow.keras.callbacks import Callback
import numpy as np
from sklearn.metrics import f1_score

class WeightedAccuracyCallback(Callback):
    def __init__(self, X_val, key_val,
                 y_root_val, y_quality_val, y_ext_val,
                 chord_vocab, key_vocab,
                 w_root=0.6, w_quality=0.3, w_ext=0.1):
        super().__init__()

        self.X_val = X_val
        self.key_val = key_val

        self.y_root_val = y_root_val.reshape(-1)
        self.y_quality_val = y_quality_val.reshape(-1)
        self.y_ext_val = y_ext_val.reshape(-1)

        self.idx_to_chord = {i:c for i,c in enumerate(chord_vocab)}
        self.idx_to_key = {i:k for i,k in enumerate(key_vocab)}

        s = w_root + w_quality + w_ext
        self.wr, self.wq, self.we = w_root/s, w_quality/s, w_ext/s
        self.history = {
    "weighted_acc": [],
    "macro_root": [],
    "top3_root": [],
    "f1_root": [],
    "functional_acc": [],
    "ii_v_i": []
}

    # Helper metrics
    def macro_accuracy(self, y_true, y_pred, num_classes):
        accs = []
        for c in range(num_classes):
            mask = (y_true == c)
            if np.sum(mask) == 0:
                continue
            accs.append(np.mean(y_pred[mask] == y_true[mask]))
        return np.mean(accs)

    def top_k_accuracy(self, y_true, probs, k=3):
        top_k = np.argsort(probs, axis=1)[:, -k:]
        return np.mean([y_true[i] in top_k[i] for i in range(len(y_true))])

    def root_to_function(self, r):
        if r == 0:
            return 0  # tonic
        elif r in [2, 5]:
            return 1  # subdominant
        elif r in [7, 11]:
            return 2  # dominant
        else:
            return 3  # other

    def functional_accuracy(self, y_true, y_pred):
        y_true_f = np.array([self.root_to_function(r) for r in y_true])
        y_pred_f = np.array([self.root_to_function(r) for r in y_pred])
        return np.mean(y_true_f == y_pred_f)

    def ii_V_I_accuracy(self, X_val, key_val, pred_root):
        total, correct = 0, 0

        for i in range(len(X_val)):
            window = X_val[i]
            key = self.idx_to_key[key_val[i]]

            chords = [self.idx_to_chord[idx] for idx in window]
            if len(chords) < 2:
                continue

            c1, c2 = chords[-2], chords[-1]
            r1 = chord_to_rel_root(c1, key)
            r2 = chord_to_rel_root(c2, key)

            if r1 == 2 and r2 == 7:
                total += 1
                if pred_root[i] == 0:
                    correct += 1

        return correct / total if total > 0 else np.nan

    # Main callback
    def on_epoch_end(self, epoch, logs=None):
        pr, pq, pe = self.model.predict(
            {"chord_window": self.X_val, "song_key": self.key_val},
            verbose=0
        )

        yhr = np.argmax(pr, axis=1)
        yhq = np.argmax(pq, axis=1)
        yhe = np.argmax(pe, axis=1)

        # Weighted accuracy
        wacc = (
            self.wr*(yhr==self.y_root_val) +
            self.wq*(yhq==self.y_quality_val) +
            self.we*(yhe==self.y_ext_val)
        ).mean()

        # Macro accuracy
        macro_root = self.macro_accuracy(self.y_root_val, yhr, 13)
        macro_quality = self.macro_accuracy(self.y_quality_val, yhq, len(np.unique(self.y_quality_val)))
        macro_ext = self.macro_accuracy(self.y_ext_val, yhe, len(np.unique(self.y_ext_val)))

        # Top-k
        top3_root = self.top_k_accuracy(self.y_root_val, pr, k=3)

        # F1
        f1_root = f1_score(self.y_root_val, yhr, average="weighted")
        f1_quality = f1_score(self.y_quality_val, yhq, average="weighted")
        f1_ext = f1_score(self.y_ext_val, yhe, average="weighted")

        # Functional
        func_acc = self.functional_accuracy(self.y_root_val, yhr)

        # ii–V–I
        ii_v_i = self.ii_V_I_accuracy(self.X_val, self.key_val, yhr)

        logs = logs or {}
        logs["val_weighted_accuracy"] = float(wacc)

        print(f"""
 Validation Metrics:
Weighted Acc: {wacc:.4f}
Macro Root: {macro_root:.4f} | Quality: {macro_quality:.4f} | Ext: {macro_ext:.4f}
Top-3 Root: {top3_root:.4f}
F1 Root: {f1_root:.4f} | Quality: {f1_quality:.4f} | Ext: {f1_ext:.4f}
Functional Acc: {func_acc:.4f}
ii–V→I Acc: {ii_v_i:.4f}
""")
        self.last_metrics = {
    "weighted_acc": float(wacc),
    "macro_root": float(macro_root),
    "top3_root": float(top3_root),
    "f1_root": float(f1_root),
    "functional_acc": float(func_acc),
    "ii_v_i": float(ii_v_i),
}
        self.history["weighted_acc"].append(float(wacc))
        self.history["macro_root"].append(float(macro_root))
        self.history["top3_root"].append(float(top3_root))
        self.history["f1_root"].append(float(f1_root))
        self.history["functional_acc"].append(float(func_acc))
        self.history["ii_v_i"].append(float(ii_v_i))

In [ ]:
from tensorflow.keras import layers, Model, regularizers



def make_model(WINDOW, chord_vocab_size, key_vocab_size, num_roots, num_qualities, num_exts,
               embedding_dim=64, key_emb_dim=32, lstm_units=64, dropout=0.6, rec_dropout=0.4):
    l2_reg = regularizers.l2(1e-4)
    chord_in = layers.Input(shape=(WINDOW,), dtype="int32", name="chord_window")
    key_in   = layers.Input(shape=(), dtype="int32", name="song_key")

    x = layers.Embedding(chord_vocab_size, embedding_dim, name="chord_embedding")(chord_in)
    x = layers.Dropout(0.3)(x)
    x = layers.LSTM(lstm_units, dropout=dropout, recurrent_dropout=rec_dropout, name="shared_lstm")(x)

    k = layers.Embedding(key_vocab_size, key_emb_dim, name="key_embedding")(key_in)

    h = layers.Concatenate(name="fuse")([x, k])

    root_out = layers.Dense(num_roots, activation="softmax", kernel_regularizer=l2_reg, name="root")(h)
    quality_branch = layers.Dense(32,activation="relu", kernel_regularizer=l2_reg, name="quality_refiner")(h)
    quality_out = layers.Dense(num_qualities, activation="softmax", kernel_regularizer=l2_reg, name="quality")(quality_branch)
    ext_out = layers.Dense(num_exts, activation="softmax", kernel_regularizer=l2_reg, name="extension")(h)

    model = Model([chord_in, key_in], [root_out, quality_out, ext_out], name="ChordFactorLSTM_CV")

    model.compile(
        optimizer="adam",
        loss={"root": "sparse_categorical_crossentropy",
              "quality": "sparse_categorical_crossentropy",
              "extension": "sparse_categorical_crossentropy"},
        loss_weights={"root": 0.6, "quality": 0.3, "extension": 0.1},
        metrics={"root": ["accuracy"], "quality": ["accuracy"], "extension": ["accuracy"]},
    )
    return model

In [ ]:
import numpy as np
import random
import tensorflow as tf
from sklearn.model_selection import KFold

N_FOLDS = 30
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

num_songs = len(all_chord_sequences)
song_indices = np.arange(num_songs)

print("Songs:", num_songs)

Songs: 113


In [ ]:
def evaluate_ii_V_I(
    model,
    X_val,
    key_val,
    y_root_val,
    chord_vocab,
    key_vocab,
    WINDOW
):

    idx_to_chord = {i:c for i,c in enumerate(chord_vocab)}
    idx_to_key = {i:k for i,k in enumerate(key_vocab)}

    pr, _, _ = model.predict(
        {"chord_window": X_val, "song_key": key_val},
        verbose=0
    )
    pred_root = np.argmax(pr, axis=1)

    total = 0
    correct = 0

    for i in range(len(X_val)):
        window = X_val[i]
        key = idx_to_key[key_val[i]]

        chords = [idx_to_chord[idx] for idx in window]
        if len(chords) < 2:
            continue

        c1, c2 = chords[-2], chords[-1]

        r1 = chord_to_rel_root(c1, key)
        r2 = chord_to_rel_root(c2, key)

        if r1 == 2 and r2 == 7:
            total += 1

            if pred_root[i] == 0:
                correct += 1

    if total == 0:
        print("No ii–V patterns found in validation set.")
        return

    print(f"ii–V → I accuracy: {correct}/{total} = {correct/total:.4f}")

In [ ]:
import numpy as np
from collections import Counter, defaultdict

def combine_chords(root, quality, ext):
    return list(zip(root, quality, ext))

def majority_component_baseline(
    y_root_train, y_quality_train, y_ext_train,
    y_root_val, y_quality_val, y_ext_val
):
    root_major = Counter(y_root_train).most_common(1)[0][0]
    qual_major = Counter(y_quality_train).most_common(1)[0][0]
    ext_major  = Counter(y_ext_train).most_common(1)[0][0]

    root_pred = np.full_like(y_root_val, root_major)
    qual_pred = np.full_like(y_quality_val, qual_major)
    ext_pred  = np.full_like(y_ext_val, ext_major)

    root_acc = np.mean(root_pred == y_root_val)
    qual_acc = np.mean(qual_pred == y_quality_val)
    ext_acc  = np.mean(ext_pred  == y_ext_val)

    return {
        "maj_root_acc": root_acc,
        "maj_quality_acc": qual_acc,
        "maj_ext_acc": ext_acc
    }


def train_markov_components(X_train, y_root, y_quality, y_ext):
    root_trans = defaultdict(Counter)
    qual_trans = defaultdict(Counter)
    ext_trans  = defaultdict(Counter)

    for i in range(len(X_train)):
        prev_root = X_train[i][-1]

        root_trans[prev_root][y_root[i]] += 1
        qual_trans[prev_root][y_quality[i]] += 1
        ext_trans[prev_root][y_ext[i]] += 1

    def normalize(trans):
        model = {}
        for k, counter in trans.items():
            total = sum(counter.values())
            model[k] = {kk: vv / total for kk, vv in counter.items()}
        return model

    return normalize(root_trans), normalize(qual_trans), normalize(ext_trans)


def predict_from_markov(model, prev_root, fallback):
    if prev_root in model:
        return max(model[prev_root], key=model[prev_root].get)
    return fallback


def markov_component_baseline(
    X_train,
    y_root_train, y_quality_train, y_ext_train,
    X_val,
    y_root_val, y_quality_val, y_ext_val
):
    root_m, qual_m, ext_m = train_markov_components(
        X_train, y_root_train, y_quality_train, y_ext_train
    )

    # fallbacks
    root_fb = Counter(y_root_train).most_common(1)[0][0]
    qual_fb = Counter(y_quality_train).most_common(1)[0][0]
    ext_fb  = Counter(y_ext_train).most_common(1)[0][0]

    root_preds = []
    qual_preds = []
    ext_preds  = []

    for i in range(len(X_val)):
        prev_root = X_val[i][-1]

        root_preds.append(predict_from_markov(root_m, prev_root, root_fb))
        qual_preds.append(predict_from_markov(qual_m, prev_root, qual_fb))
        ext_preds.append(predict_from_markov(ext_m, prev_root, ext_fb))

    root_preds = np.array(root_preds)
    qual_preds = np.array(qual_preds)
    ext_preds  = np.array(ext_preds)

    return {
        "markov_root_acc": np.mean(root_preds == y_root_val),
        "markov_quality_acc": np.mean(qual_preds == y_quality_val),
        "markov_ext_acc": np.mean(ext_preds == y_ext_val)
    }

def evaluate_component_baselines(
    X_train,
    y_root_train, y_quality_train, y_ext_train,
    X_val,
    y_root_val, y_quality_val, y_ext_val
):

    maj = majority_component_baseline(
        y_root_train, y_quality_train, y_ext_train,
        y_root_val, y_quality_val, y_ext_val
    )

    mkv = markov_component_baseline(
        X_train,
        y_root_train, y_quality_train, y_ext_train,
        X_val,
        y_root_val, y_quality_val, y_ext_val
    )

    print("Component Baselines: ")
    print(f"Majority: {maj}")
    print(f"Markov: {mkv}")

    return {**maj, **mkv}

In [ ]:
def exact_chord_accuracy(model, X_val, key_val,
                         y_root_val, y_quality_val, y_ext_val):

    # Get predictions
    root_pred, quality_pred, ext_pred = model.predict([X_val, key_val], verbose=0)

    # Convert to class indices
    root_pred = np.argmax(root_pred, axis=1)
    quality_pred = np.argmax(quality_pred, axis=1)
    ext_pred = np.argmax(ext_pred, axis=1)

    # Exact match across ALL components
    correct = (
        (root_pred == y_root_val) &
        (quality_pred == y_quality_val) &
        (ext_pred == y_ext_val)
    )

    acc = np.mean(correct)
    return acc

In [ ]:
def root_accuracy(model, X_val, key_val, y_root_val):
    root_pred = model.predict([X_val, key_val], verbose=0)[0]
    root_pred = np.argmax(root_pred, axis=1)

    return np.mean(root_pred == y_root_val)

In [ ]:
# Running this cell will take 2+ hours - run at your own risk!


from tensorflow.keras.callbacks import EarlyStopping

WINDOW = 24
EPOCHS = 15
BATCH = 64

fold_results = []
fold_histories = []

for fold, (train_ids, val_ids) in enumerate(kf.split(song_indices), start=1):

    tf.keras.utils.set_random_seed(SEED + fold)

    train_song_ids = song_indices[train_ids]
    val_song_ids   = song_indices[val_ids]

    aug_chord_sequences = []
    aug_song_keys = []

    for sid in train_song_ids:
        aug_chord_sequences.append(all_chord_sequences[sid])
        aug_song_keys.append(song_keys[sid])

    for sid in train_song_ids:
        seq = all_chord_sequences[sid]
        key = song_keys[sid]

        for shift in range(1, 3):  # 2 transpositions
            new_seq = [transpose_chord(c, shift) for c in seq]
            new_key = transpose_key(key, shift)
            aug_chord_sequences.append(new_seq)
            aug_song_keys.append(new_key)

    (X_train, key_train, y_root_train, y_quality_train, y_ext_train,
     X_val, key_val, y_root_val, y_quality_val, y_ext_val, meta, chord_vocab, key_vocab) = \
        build_fold_data_from_lists(
            aug_chord_sequences,
            aug_song_keys,
            val_song_ids,
            WINDOW
        )

    print(f"\n================ Fold {fold}/{N_FOLDS} ================")
    print("Train windows:", len(X_train), "Val windows:", len(X_val))
    print("Vocab sizes:", meta)

    model = make_model(
        WINDOW,
        chord_vocab_size=meta["chord_vocab_size"],
        key_vocab_size=meta["key_vocab_size"],
        num_roots=meta["num_roots"],
        num_qualities=meta["num_qualities"],
        num_exts=meta["num_exts"],
        embedding_dim=64, key_emb_dim=32, lstm_units=64
    )

    weighted_cb = WeightedAccuracyCallback(
    X_val, key_val,
    y_root_val, y_quality_val, y_ext_val,
    chord_vocab, key_vocab,
    w_root=0.6, w_quality=0.3, w_ext=0.1
)

    early = EarlyStopping(
        monitor="val_weighted_accuracy",
        mode="max",
        patience=5,
        restore_best_weights=True
    )

    hist = model.fit(
        {"chord_window": X_train, "song_key": key_train},
        {"root": y_root_train, "quality": y_quality_train, "extension": y_ext_train},
        validation_data=(
            {"chord_window": X_val, "song_key": key_val},
            {"root": y_root_val, "quality": y_quality_val, "extension": y_ext_val}
        ),
        epochs=EPOCHS,
        batch_size=BATCH,
        callbacks=[weighted_cb, early],
        verbose=1
    )
    fold_histories.append(weighted_cb.history)

    metrics = weighted_cb.last_metrics

    best_val_weighted_accuracy = max(hist.history.get("val_weighted_accuracy", [np.nan]))
    best_val_root_acc = max(hist.history.get("val_root_accuracy", [np.nan]))
    best_val_quality_acc = max(hist.history.get("val_quality_accuracy", [np.nan]))
    best_val_ext_acc = max(hist.history.get("val_extension_accuracy", [np.nan]))

    print(f"\nModel Best Validation Accuracies for Fold {fold}:")
    print(f"  Weighted Accuracy: {best_val_weighted_accuracy:.4f}")
    print(f"  Root Accuracy:     {best_val_root_acc:.4f}")
    print(f"  Quality Accuracy:  {best_val_quality_acc:.4f}")
    print(f"  Extension Accuracy: {best_val_ext_acc:.4f}")

    evaluate_ii_V_I(
        model,
        X_val,
        key_val,
        y_root_val,
        chord_vocab,
        key_vocab,
        WINDOW
    )

    baseline_results = evaluate_component_baselines(
        X_train,
        y_root_train, y_quality_train, y_ext_train,
        X_val,
        y_root_val, y_quality_val, y_ext_val
    )

    exact_acc = exact_chord_accuracy(
        model, X_val, key_val,
        y_root_val, y_quality_val, y_ext_val
    )
    root_acc = root_accuracy(
        model, X_val, key_val,
        y_root_val
    )
    print(f"Exact Chord Accuracy: {exact_acc:.4f}")
    print(f"Root Accuracy:        {root_acc:.4f}")

    fold_results.append({
        "fold": fold,
        **metrics,
        "best_val_weighted_accuracy": best_val_weighted_accuracy,
        "best_val_root_acc": best_val_root_acc,
        "best_val_quality_acc": best_val_quality_acc,
        "best_val_ext_acc": best_val_ext_acc,
        "train_windows": len(X_train),
        "val_windows": len(X_val),
        "exact_chord_acc": exact_acc,
        "root_acc": root_acc,
        **baseline_results
    })



================ Fold 1/30 ================
Train windows: 60690 Val windows: 593
Vocab sizes: {'chord_vocab_size': 589, 'key_vocab_size': 29, 'num_qualities': 9, 'num_exts': 8, 'num_roots': 13}
Epoch 1/15
949/949 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - extension_accuracy: 0.5479 - extension_loss: 1.2275 - loss: 1.6809 - quality_accuracy: 0.4515 - quality_loss: 1.4164 - root_accuracy: 0.4193 - root_loss: 1.8722
 Validation Metrics:
Weighted Acc: 0.3882
Macro Root: 0.1425 | Quality: 0.2927 | Ext: 0.2422
Top-3 Root: 0.5093
F1 Root: 0.1831 | Quality: 0.4717 | Ext: 0.4976
Functional Acc: 0.3558
ii–V→I Acc: 0.9057

949/949 ━━━━━━━━━━━━━━━━━━━━ 64s 60ms/step - extension_accuracy: 0.5902 - extension_loss: 1.0701 - loss: 1.5142 - quality_accuracy: 0.4850 - quality_loss: 1.2656 - root_accuracy: 0.4603 - root_loss: 1.6959 - val_extension_accuracy: 0.6138 - val_extension_loss: 1.2224 - val_loss: 1.9682 - val_quality_accuracy: 0.5329 - val_quality_loss: 1.3160 - val_root_accuracy: 0.2782 - val_root_l

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


956/956 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - extension_accuracy: 0.6300 - extension_loss: 0.9172 - loss: 1.3671 - quality_accuracy: 0.5514 - quality_loss: 1.1140 - root_accuracy: 0.4988 - root_loss: 1.5509
 Validation Metrics:
Weighted Acc: 0.4433
Macro Root: 0.2136 | Quality: 0.3849 | Ext: nan
Top-3 Root: 0.7373
F1 Root: 0.3179 | Quality: 0.3800 | Ext: 0.5513
Functional Acc: 0.4378
ii–V→I Acc: 0.8235

956/956 ━━━━━━━━━━━━━━━━━━━━ 58s 61ms/step - extension_accuracy: 0.6354 - extension_loss: 0.9027 - loss: 1.3458 - quality_accuracy: 0.5578 - quality_loss: 1.0971 - root_accuracy: 0.5055 - root_loss: 1.5256 - val_extension_accuracy: 0.5553 - val_extension_loss: 0.8255 - val_loss: 1.5702 - val_quality_accuracy: 0.4217 - val_quality_loss: 1.2818 - val_root_accuracy: 0.4355 - val_root_loss: 1.8146 - val_weighted_accuracy: 0.4433
Epoch 3/15
  2/956 ━━━━━━━━━━━━━━━━━━━━ 56s 59ms/step - extension_accuracy: 0.5977 - extension_loss: 0.9139 - loss: 1.2969 - quality_accuracy: 0.5781 - quality_loss: 

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


956/956 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - extension_accuracy: 0.6532 - extension_loss: 0.8520 - loss: 1.2897 - quality_accuracy: 0.5834 - quality_loss: 1.0427 - root_accuracy: 0.5283 - root_loss: 1.4661
 Validation Metrics:
Weighted Acc: 0.4558
Macro Root: 0.2247 | Quality: 0.3701 | Ext: nan
Top-3 Root: 0.7028
F1 Root: 0.3349 | Quality: 0.4137 | Ext: 0.5210
Functional Acc: 0.4424
ii–V→I Acc: 0.8235

956/956 ━━━━━━━━━━━━━━━━━━━━ 58s 61ms/step - extension_accuracy: 0.6552 - extension_loss: 0.8484 - loss: 1.2746 - quality_accuracy: 0.5863 - quality_loss: 1.0324 - root_accuracy: 0.5336 - root_loss: 1.4462 - val_extension_accuracy: 0.5207 - val_extension_loss: 0.8077 - val_loss: 1.5820 - val_quality_accuracy: 0.4654 - val_quality_loss: 1.2800 - val_root_accuracy: 0.4401 - val_root_loss: 1.8344 - val_weighted_accuracy: 0.4558
Epoch 4/15
  2/956 ━━━━━━━━━━━━━━━━━━━━ 55s 58ms/step - extension_accuracy: 0.6250 - extension_loss: 0.8938 - loss: 1.2668 - quality_accuracy: 0.6016 - quality_loss: 

...

956/956 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - extension_accuracy: 0.6844 - extension_loss: 0.7750 - loss: 1.1087 - quality_accuracy: 0.6418 - quality_loss: 0.9075 - root_accuracy: 0.5938 - root_loss: 1.2366
 Validation Metrics:
Weighted Acc: 0.4634
Macro Root: 0.2455 | Quality: 0.3976 | Ext: nan
Top-3 Root: 0.6797
F1 Root: 0.3724 | Quality: 0.4229 | Ext: 0.5207
Functional Acc: 0.4839
ii–V→I Acc: 0.8235

956/956 ━━━━━━━━━━━━━━━━━━━━ 58s 61ms/step - extension_accuracy: 0.6851 - extension_loss: 0.7767 - loss: 1.1060 - quality_accuracy: 0.6420 - quality_loss: 0.9073 - root_accuracy: 0.5947 - root_loss: 1.2317 - val_extension_accuracy: 0.5207 - val_extension_loss: 0.8447 - val_loss: 1.6213 - val_quality_accuracy: 0.4677 - val_quality_loss: 1.2727 - val_root_accuracy: 0.4516 - val_root_loss: 1.8888 - val_weighted_accuracy: 0.4634
Epoch 9/15
  2/956 ━━━━━━━━━━━━━━━━━━━━ 1:05 69ms/step - extension_accuracy: 0.6758 - extension_loss: 0.8024 - loss: 1.1110 - quality_accuracy: 0.6523 - quality_loss:

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


956/956 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - extension_accuracy: 0.6884 - extension_loss: 0.7661 - loss: 1.0857 - quality_accuracy: 0.6468 - quality_loss: 0.8924 - root_accuracy: 0.6022 - root_loss: 1.2059
 Validation Metrics:
Weighted Acc: 0.4502
Macro Root: 0.2380 | Quality: 0.4004 | Ext: nan
Top-3 Root: 0.6774
F1 Root: 0.3512 | Quality: 0.4131 | Ext: 0.5207
Functional Acc: 0.4793
ii–V→I Acc: 0.8235

956/956 ━━━━━━━━━━━━━━━━━━━━ 58s 61ms/step - extension_accuracy: 0.6868 - extension_loss: 0.7695 - loss: 1.0849 - quality_accuracy: 0.6461 - quality_loss: 0.8940 - root_accuracy: 0.6004 - root_loss: 1.2029 - val_extension_accuracy: 0.5207 - val_extension_loss: 0.8554 - val_loss: 1.6176 - val_quality_accuracy: 0.4516 - val_quality_loss: 1.2651 - val_root_accuracy: 0.4378 - val_root_loss: 1.8831 - val_weighted_accuracy: 0.4502


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



Model Best Validation Accuracies for Fold 28:
  Weighted Accuracy: 0.4832
  Root Accuracy:     0.4631
  Quality Accuracy:  0.5000
  Extension Accuracy: 0.5553
ii–V → I accuracy: 28/34 = 0.8235
Component Baselines: 
Majority: {'maj_root_acc': np.float64(0.33640552995391704), 'maj_quality_acc': np.float64(0.30414746543778803), 'maj_ext_acc': np.float64(0.45852534562211983)}
Markov: {'markov_root_acc': np.float64(0.4308755760368664), 'markov_quality_acc': np.float64(0.41244239631336405), 'markov_ext_acc': np.float64(0.47465437788018433)}
Exact Chord Accuracy: 0.2788
Root Accuracy:        0.4631

================ Fold 29/30 ================
Train windows: 60225 Val windows: 748
Vocab sizes: {'chord_vocab_size': 626, 'key_vocab_size': 28, 'num_qualities': 9, 'num_exts': 8, 'num_roots': 13}
Epoch 1/15
942/942 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - extension_accuracy: 0.5560 - extension_loss: 1.2352 - loss: 1.6901 - quality_accuracy: 0.4512 - quality_loss: 1.4055 - root_accuracy: 0.4125 - root_

In [ ]:
def summarize_metric(name):
    vals = []
    for f in fold_results:
        if name in f and not np.isnan(f[name]):
            vals.append(f[name])
        elif "baseline_results" in f and name in f["baseline_results"] and not np.isnan(f["baseline_results"][name]):
            vals.append(f["baseline_results"][name])

    if len(vals) == 0:
        print(f"Warning: No values found for {name}")
        return np.nan, np.nan

    return np.mean(vals), np.std(vals)


metrics_to_report = [
    "weighted_acc",
    "best_val_weighted_accuracy",
    "best_val_root_acc",
    "best_val_quality_acc",
    "best_val_ext_acc",
    "macro_root",
    "top3_root",
    "f1_root",
    "functional_acc",
    "ii_v_i",
    "maj_root_acc",
    "maj_quality_acc",
    "maj_ext_acc",
    "markov_root_acc",
    "markov_quality_acc",
    "markov_ext_acc"
]

print("\n Final Results: ")

for m in metrics_to_report:
    mean, std = summarize_metric(m)
    print(f"{m}: {mean:.4f} \u00b1 {std:.4f}")


 Final Results: 
weighted_acc: 0.4597 ± 0.0900
best_val_weighted_accuracy: 0.4866 ± 0.0845
best_val_root_acc: 0.4509 ± 0.1376
best_val_quality_acc: 0.5397 ± 0.0763
best_val_ext_acc: 0.6257 ± 0.0946
macro_root: 0.2032 ± 0.0715
top3_root: 0.6524 ± 0.1122
f1_root: 0.3758 ± 0.1592
functional_acc: 0.4649 ± 0.1291
ii_v_i: 0.7381 ± 0.2050
maj_root_acc: 0.2584 ± 0.0668
maj_quality_acc: 0.4259 ± 0.0888
maj_ext_acc: 0.5497 ± 0.1040
markov_root_acc: 0.3142 ± 0.0759
markov_quality_acc: 0.4716 ± 0.0752
markov_ext_acc: 0.5877 ± 0.0977
